# NanoBurnGPT on Google Colab GPUs

Train and run **NanoBurnGPT** (pure Rust + [Burn 0.22](https://burn.dev/) + CUDA) directly on a Google Colab GPU.

### Before you start
1. Select **Runtime -> Change runtime type -> GPU** (T4, L4, or A100).
2. **Notes on Burn 0.22 + Colab:**
   - Training currently runs in **`f32`** (`--amp` is a no-op on Burn 0.22). On a 15 GB T4, use a smaller `--batch-size` / `--block-size` with `--grad-accum-steps` to avoid OOM.
   - The repo sets `lto = "fat"` and `codegen-units = 1` in `Cargo.toml`. In Colab (~12.7 GB system RAM), Step 4 overrides those with `CARGO_PROFILE_RELEASE_LTO=thin` and `CARGO_PROFILE_RELEASE_CODEGEN_UNITS=16` and disables the `tui` + `wgpu` features (`--no-default-features --features cuda`).
   - Enable Google Drive persistence in Step 2 to keep checkpoints (`artifacts/`) and cache the compiled CUDA binary across Colab restarts.
   - If your Burn 0.22 work lives on a feature branch rather than `main`, set `BRANCH` in Step 3 to that pushed branch name.

In [ ]:
# Step 1: Verify GPU & CUDA availability
import shutil
import subprocess

res = subprocess.run(["nvidia-smi"], text=True)
if res.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU detected. Go to Runtime -> Change runtime type -> GPU."
    )

if shutil.which("nvcc"):
    subprocess.run(["nvcc", "--version"], check=False)
else:
    print("Note: nvcc not found on PATH; cubecl-cuda uses NVRTC/driver dynamic loading.")


In [ ]:
# Step 2: Optional Google Drive persistence
import os
from pathlib import Path

USE_DRIVE = True #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/nanoburngpt" #@param {type:"string"}

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running inside Google Colab; using local directory for persistence.")
    Path(DRIVE_DIR, "artifacts").mkdir(parents=True, exist_ok=True)
    Path(DRIVE_DIR, "bin").mkdir(parents=True, exist_ok=True)
    print(f"Persistence enabled at: {DRIVE_DIR}")
else:
    print("Google Drive persistence disabled; artifacts will live in ephemeral /content storage.")


In [ ]:
# Step 3: Clone repository & install Rust toolchain
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tmccrmck/nanoburngpt.git" #@param {type:"string"}
BRANCH = "burn-0.22-colab" #@param {type:"string"}
REPO_DIR = Path("/content/nanoburngpt")


def run_streamed(cmd, env=None, char_stream=False):
    """Run a command and stream stdout/stderr live into the notebook cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1 if not char_stream else 0,
        env=env,
    )
    assert proc.stdout is not None
    if char_stream:
        while True:
            ch = proc.stdout.read(1)
            if ch == "" and proc.poll() is not None:
                break
            if ch:
                sys.stdout.write(ch)
                sys.stdout.flush()
    else:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


# Clone or update repository
if not REPO_DIR.exists():
    clone_cmd = ["git", "clone"]
    if BRANCH.strip():
        clone_cmd += ["--branch", BRANCH.strip()]
    clone_cmd += [REPO_URL, str(REPO_DIR)]
    run_streamed(clone_cmd)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    if BRANCH.strip():
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH.strip()], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
subprocess.run(["git", "log", "-n", "3", "--oneline"], check=True)

# Symlink artifacts/ to Google Drive when enabled
use_drive = globals().get("USE_DRIVE", False)
drive_dir = globals().get("DRIVE_DIR", "/content/drive/MyDrive/nanoburngpt")
if use_drive:
    drive_artifacts = Path(drive_dir) / "artifacts"
    drive_artifacts.mkdir(parents=True, exist_ok=True)
    local_artifacts = REPO_DIR / "artifacts"
    if local_artifacts.is_symlink():
        if local_artifacts.resolve() != drive_artifacts.resolve():
            local_artifacts.unlink()
            local_artifacts.symlink_to(drive_artifacts)
    elif local_artifacts.exists():
        if any(local_artifacts.iterdir()):
            backup = REPO_DIR / "artifacts.local.bak"
            if backup.exists():
                shutil.rmtree(backup)
            local_artifacts.rename(backup)
            print(f"Moved existing non-empty artifacts/ to {backup}")
        else:
            local_artifacts.rmdir()
        local_artifacts.symlink_to(drive_artifacts)
    else:
        local_artifacts.symlink_to(drive_artifacts)
    print(f"artifacts/ -> {drive_artifacts}")

# Install Rust via rustup if needed
cargo_bin = str(Path.home() / ".cargo" / "bin")
if cargo_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = cargo_bin + os.pathsep + os.environ.get("PATH", "")

if not shutil.which("cargo"):
    print("Installing Rust via rustup...")
    run_streamed([
        "sh",
        "-c",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
    ])

# Running inside REPO_DIR triggers rust-toolchain.toml auto-install if present
run_streamed(["rustc", "--version"])
run_streamed(["cargo", "--version"])


In [ ]:
# Step 4: Build release binary with CUDA (Colab-friendly LTO + optional Drive cache)
import os
import shutil
import subprocess
from pathlib import Path

CACHE_BINARY_IN_DRIVE = True #@param {type:"boolean"}
FORCE_REBUILD = False #@param {type:"boolean"}

head_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
target_bin = Path("target/release/nanoburngpt")
use_drive = globals().get("USE_DRIVE", False)
drive_dir = globals().get("DRIVE_DIR", "/content/drive/MyDrive/nanoburngpt")
cached_bin = Path(drive_dir) / "bin" / f"nanoburngpt-cuda-{head_sha}"

if use_drive and CACHE_BINARY_IN_DRIVE and not FORCE_REBUILD and cached_bin.exists():
    target_bin.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached_bin, target_bin)
    target_bin.chmod(0o755)
    print(f"Restored cached CUDA binary from {cached_bin}")
else:
    build_env = os.environ.copy()
    # Keep linking memory within Colab's ~12.7 GB system RAM limit
    build_env["CARGO_PROFILE_RELEASE_LTO"] = "thin"
    build_env["CARGO_PROFILE_RELEASE_CODEGEN_UNITS"] = "16"
    print(f"Building commit {head_sha} with --no-default-features --features cuda ...")
    run_streamed(
        ["cargo", "build", "--release", "--no-default-features", "--features", "cuda"],
        env=build_env,
    )
    if use_drive and CACHE_BINARY_IN_DRIVE:
        cached_bin.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target_bin, cached_bin)
        print(f"Cached binary to {cached_bin}")


In [ ]:
# Step 5: Quick smoke test (optional)
run_streamed([
    "./target/release/nanoburngpt",
    "train",
    "--model", "nano",
    "--dataset", "shakespeare",
    "--batch-size", "16",
    "--max-train-items", "2000",
    "--num-epochs", "1",
    "--num-workers", "2",
    "--no-tui",
])

run_streamed(
    ["./target/release/nanoburngpt", "generate", "--max-tokens", "50"],
    char_stream=True,
)


In [ ]:
# Step 6: Train on GPU
MODEL = "gpt2-small" #@param ["nano", "gpt2-small", "gpt2-medium", "gpt2-large", "gpt2-xl"]
DATASET = "shakespeare" #@param ["shakespeare", "wikitext103"]
BATCH_SIZE = 8 #@param {type:"integer"}
GRAD_ACCUM_STEPS = 4 #@param {type:"integer"}
# 0 = use preset default. Note: training runs in f32; 256-512 fits a 15 GB T4,
# while an A100 can typically use 0 (preset default 1024).
BLOCK_SIZE = 256 #@param {type:"integer"}
NUM_EPOCHS = 5 #@param {type:"integer"}
LEARNING_RATE = 3e-4 #@param {type:"number"}
WARMUP_ITERS = 100 #@param {type:"integer"}
NUM_WORKERS = 2 #@param {type:"integer"}
# 0 = full dataset
MAX_TRAIN_ITEMS = 0 #@param {type:"integer"}

cmd = [
    "./target/release/nanoburngpt",
    "train",
    "--model", MODEL,
    "--dataset", DATASET,
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--num-epochs", str(NUM_EPOCHS),
    "--learning-rate", str(LEARNING_RATE),
    "--warmup-iters", str(WARMUP_ITERS),
    "--num-workers", str(NUM_WORKERS),
    "--no-tui",
]
if BLOCK_SIZE > 0:
    cmd += ["--block-size", str(BLOCK_SIZE)]
if MAX_TRAIN_ITEMS > 0:
    cmd += ["--max-train-items", str(MAX_TRAIN_ITEMS)]

print("Running:", " ".join(cmd))
run_streamed(cmd)


In [ ]:
# Step 7: Plot Loss & Perplexity curves from artifacts/
import re
from pathlib import Path
import matplotlib.pyplot as plt


def load_epoch_means(split: str, metric: str):
    split_dir = Path("artifacts") / split
    if not split_dir.exists():
        return [], []

    epoch_dirs = []
    for entry in split_dir.iterdir():
        m = re.fullmatch(r"epoch-(\d+)", entry.name)
        if m and entry.is_dir():
            epoch_dirs.append((int(m.group(1)), entry))
    epoch_dirs.sort(key=lambda item: item[0])

    epochs, means = [], []
    for epoch, epoch_dir in epoch_dirs:
        log_path = epoch_dir / f"{metric}.log"
        if not log_path.exists():
            continue
        weighted_sum = 0.0
        total_weight = 0.0
        for line in log_path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) != 2:
                continue
            val, count = float(parts[0]), float(parts[1])
            weighted_sum += val * count
            total_weight += count
        if total_weight > 0:
            epochs.append(epoch)
            means.append(weighted_sum / total_weight)
    return epochs, means


train_loss_ep, train_loss = load_epoch_means("train", "Loss")
valid_loss_ep, valid_loss = load_epoch_means("valid", "Loss")
train_ppl_ep, train_ppl = load_epoch_means("train", "Perplexity")
valid_ppl_ep, valid_ppl = load_epoch_means("valid", "Perplexity")

if not (train_loss or valid_loss or train_ppl or valid_ppl):
    print("No metric logs found under artifacts/train or artifacts/valid yet. Run training first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    if train_loss:
        axes[0].plot(train_loss_ep, train_loss, marker="o", label="Train")
    if valid_loss:
        axes[0].plot(valid_loss_ep, valid_loss, marker="o", label="Valid")
    axes[0].set_title("Cross-Entropy Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    if train_ppl:
        axes[1].plot(train_ppl_ep, train_ppl, marker="o", label="Train")
    if valid_ppl:
        axes[1].plot(valid_ppl_ep, valid_ppl, marker="o", label="Valid")
    axes[1].set_title("Perplexity")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Perplexity")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# Step 8: Generate text from trained checkpoint
PROMPT = "HAMLET:\n" #@param {type:"string"}
MAX_TOKENS = 200 #@param {type:"integer"}
TEMPERATURE = 0.8 #@param {type:"number"}
TOP_K = 40 #@param {type:"integer"}
TOP_P = 0.95 #@param {type:"number"}
ARTIFACT_DIR = "artifacts" #@param {type:"string"}

# Decode literal \n in Colab string form inputs if entered as two characters
prompt_text = PROMPT.replace("\\n", "\n")

run_streamed(
    [
        "./target/release/nanoburngpt",
        "generate",
        "--artifact-dir", ARTIFACT_DIR,
        "--prompt", prompt_text,
        "--max-tokens", str(MAX_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-k", str(TOP_K),
        "--top-p", str(TOP_P),
    ],
    char_stream=True,
)
